In [1]:
import pandas as pd
import numpy as np
#data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
#!wget = $data

In [2]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [3]:
## Q1
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [4]:
## Q2
df.horsepower.mean()

np.float64(254.44601556505535)

In [7]:
##Q3
base = ['engine_displacement','horsepower','vehicle_weight']#,'model_year']
n = len(df)
n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - n_test - n_val

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)
                  
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train : n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

def prepare_X_zeor(df):
    df = df.copy()
    df['age'] = 2026 - df.model_year
    feature = base + ['age']            
    df_num = df[feature]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

def prepare_X_mean(df):
    df = df.copy()
    df['age'] = 2026 - df.model_year
    feature = base + ['age']            
    df_num = df[feature]
    df_num = df_num.fillna(df_train.horsepower.mean())
    X = df_num.values
    return X

def rmse(y,y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()
    return np.sqrt(mse)

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

X_train = prepare_X_zeor(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X_zeor(df_val)
y_pred = w0 + X_val.dot(w)

rmse_z = rmse(y_val, y_pred)
print("with zero filling")
print(round(rmse_z,3))


X_train = prepare_X_mean(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X_mean(df_val)
y_pred = w0 + X_val.dot(w)

rmse_m = rmse(y_val, y_pred)
print("with mean filling")
print(round(rmse_m,3))

with zero filling
2.205
with mean filling
2.202


In [8]:
##Q4
n = len(df)
n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - n_test - n_val

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)
                  
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train : n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

def prepare_X(df):
    df = df.copy()
    df['age'] = 2026 - df.model_year
    feature = base + ['age']            
    df_num = df[feature]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

def rmse(y,y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()
    return np.sqrt(mse)

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def train_linear_regression_reg(X, y, r = 0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]
    
for r in [0.0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    
    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    
    print(r, w0, score)

0.0 53.06442443858471 2.2052931947515564
0.01 53.040793066964866 2.2052591989702135
0.1 52.829054086714905 2.2049977040167135
1 50.80107387307622 2.206423661105027
5 43.39703805261949 2.2710822950286382
10 36.709266217698 2.4039590256914116
100 9.727073237744294 3.4282329619406324


In [9]:
##Q5

error_score = []
base = ['engine_displacement','horsepower','vehicle_weight']#,'model_year']
for myseed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = int(0.2 * n)
    n_test = int(0.2 * n)
    n_train = n - n_test - n_val
    
    np.random.seed(myseed)
    idx = np.arange(n)
    np.random.shuffle(idx)
                      
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train : n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    
    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values
    
    del df_train['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    
    def prepare_X(df):
        df = df.copy()
        df['age'] = 2026 - df.model_year
        feature = base + ['age']            
        df_num = df[feature]
        df_num = df_num.fillna(0)
        X = df_num.values
        return X
    
    def rmse(y,y_pred):
        error = y - y_pred
        se = error ** 2
        mse = se.mean()
        return np.sqrt(mse)
    
    def train_linear_regression(X, y):
        ones = np.ones(X.shape[0])
        X = np.column_stack([ones, X])
    
        XTX = X.T.dot(X)
        XTX_inv = np.linalg.inv(XTX)
        w_full = XTX_inv.dot(X.T).dot(y)
    
        return w_full[0], w_full[1:]
    
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression(X_train, y_train)
    
    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    
    error = rmse(y_val, y_pred)
    
    error_score.append(error)
    print(myseed, error)

print(error_score)
np.std(error_score).round(3)

0 2.2392041430508156
1 2.202112821083731
2 2.1634197787530844
3 2.204358876852943
4 2.2018800943317203
5 2.2457851509081364
6 2.2697212079516436
7 2.190794599935368
8 2.216611201694469
9 2.2070365928234295
[np.float64(2.2392041430508156), np.float64(2.202112821083731), np.float64(2.1634197787530844), np.float64(2.204358876852943), np.float64(2.2018800943317203), np.float64(2.2457851509081364), np.float64(2.2697212079516436), np.float64(2.190794599935368), np.float64(2.216611201694469), np.float64(2.2070365928234295)]


np.float64(0.029)

In [6]:
##Q6
n = len(df)
n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - n_test - n_val

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)
                  
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train : n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

base = ['engine_displacement','horsepower','vehicle_weight']#,'model_year']

def prepare_X(df):
    df = df.copy()
    df['age'] = 2026 - df.model_year
    feature = base + ['age']            
    df_num = df[feature]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

def rmse(y,y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()
    return np.sqrt(mse)

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def train_linear_regression_reg(X, y, r = 0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]    

df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop = True)

y_full_train = np.concatenate([y_train, y_val])
                                          
X_train = prepare_X(df_full_train)
w0, w = train_linear_regression_reg(X_train, y_full_train,r=0.001)

X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)

rmse = rmse(y_test, y_pred)
print(round(rmse,3))


2.236
